# Chapter 12 — Compound Patterns (MVC)
## *Patterns of Patterns*

**Compound Pattern:** A solution that combines two or more patterns to solve a general or recurring problem.

### Model-View-Controller (MVC)
MVC is the compound pattern covered in depth. It combines:
- **Observer** — Model notifies Views of changes.
- **Strategy** — Controller is a pluggable strategy for the View.
- **Composite** — Views can be composed of nested sub-views.

| Component | Role | Pattern used |
|---|---|---|
| **Model** | Business logic & data | Subject (Observer) |
| **View** | UI rendering | Observer; Composite for widgets |
| **Controller** | Input handling, updates model | Strategy for view |

### Book context
A DJ song request app — before MVC everything was tangled; after, the three layers are cleanly separated.

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Callable

# ════════════════════════════════════════════════════════════════
# MODEL
# ════════════════════════════════════════════════════════════════
class BeatModel:
    def __init__(self):
        self._bpm      = 90
        self._playing  = False
        self._observers: list[Callable] = []

    def register(self, cb: Callable): self._observers.append(cb)

    def _notify(self):
        for cb in self._observers: cb()

    @property
    def bpm(self): return self._bpm

    @property
    def playing(self): return self._playing

    def start(self):
        self._playing = True
        self._notify()

    def stop(self):
        self._playing = False
        self._notify()

    def set_bpm(self, bpm: int):
        self._bpm = bpm
        self._notify()

In [ ]:
# ════════════════════════════════════════════════════════════════
# VIEW
# ════════════════════════════════════════════════════════════════
class BeatView:
    def __init__(self, model: BeatModel):
        self._model = model
        model.register(self.update)   # Observer registration
        self._controller: BeatController | None = None

    def set_controller(self, c: BeatController):
        self._controller = c

    def update(self):
        status = "PLAYING" if self._model.playing else "STOPPED"
        print(f"  [View] BPM={self._model.bpm}  Status={status}")

    # User actions delegate to controller (Strategy)
    def user_start(self):            self._controller.start()
    def user_stop(self):             self._controller.stop()
    def user_increase_bpm(self):     self._controller.increase_bpm()
    def user_decrease_bpm(self):     self._controller.decrease_bpm()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONTROLLER
# ════════════════════════════════════════════════════════════════
class BeatController:
    def __init__(self, model: BeatModel, view: BeatView):
        self._model = model
        self._view  = view
        view.set_controller(self)

    def start(self):          self._model.start()
    def stop(self):           self._model.stop()
    def increase_bpm(self):   self._model.set_bpm(self._model.bpm + 5)
    def decrease_bpm(self):   self._model.set_bpm(max(40, self._model.bpm - 5))

In [ ]:
# ── Wire it all together ──────────────────────────────────────────────────────
model      = BeatModel()
view       = BeatView(model)
controller = BeatController(model, view)

print("--- Start ---")
view.user_start()

print("--- Increase BPM x3 ---")
view.user_increase_bpm()
view.user_increase_bpm()
view.user_increase_bpm()

print("--- Stop ---")
view.user_stop()

## MVC in web frameworks

Django / Flask map to MVC:

| MVC | Django |
|---|---|
| Model | `models.py` (ORM classes) |
| View | `templates/` (HTML) |
| Controller | `views.py` (request handlers) |

> Django calls its controllers "views" and its views "templates" — confusingly, it follows MVP naming.

In [ ]:
# Minimal Flask-style MVC simulation (no actual HTTP)
from dataclasses import dataclass, field
from typing import Optional

# Model
@dataclass
class Task:
    id: int
    title: str
    done: bool = False

class TaskRepository:
    def __init__(self):
        self._store: dict[int, Task] = {}
        self._next_id = 1

    def add(self, title: str) -> Task:
        t = Task(self._next_id, title)
        self._store[t.id] = t
        self._next_id += 1
        return t

    def all(self) -> list[Task]:
        return list(self._store.values())

    def complete(self, task_id: int) -> Optional[Task]:
        t = self._store.get(task_id)
        if t: t.done = True
        return t


# View (renders to console)
class TaskView:
    def render_list(self, tasks: list[Task]):
        for t in tasks:
            status = "✓" if t.done else " "
            print(f"  [{status}] {t.id}. {t.title}")


# Controller
class TaskController:
    def __init__(self, repo: TaskRepository, view: TaskView):
        self._repo = repo
        self._view = view

    def add_task(self, title: str):
        task = self._repo.add(title)
        print(f"Added: {task}")
        self._view.render_list(self._repo.all())

    def complete_task(self, task_id: int):
        self._repo.complete(task_id)
        self._view.render_list(self._repo.all())


repo = TaskRepository()
view = TaskView()
ctrl = TaskController(repo, view)

ctrl.add_task("Study design patterns")
ctrl.add_task("Practice LeetCode")
ctrl.complete_task(1)

## Interview cheat-sheet

| Question | Answer |
|---|---|
| What patterns combine to make MVC? | Observer (model→view), Strategy (controller is view's strategy), Composite (nested views). |
| MVC vs MVP vs MVVM? | MVP: view is passive, presenter does logic. MVVM: view binds to viewmodel reactively. |
| Why separate concerns? | Model reusable without UI; multiple views for same model; controller swappable. |

**Pattern family:** Compound (Architectural)